# Airbnb Nightly Price Prediction — Final Model

## Objective

Train and deploy the final Airbnb nightly price prediction model for Stockholm County.

The model uses six property features: room type, guest capacity, bedrooms, beds, district, and amenity count.

A Random Forest Regressor with logarithmic target transformation is trained using predefined hyperparameters established during previous experiments.

The complete preprocessing and regression pipeline is saved to MLflow for integration with the application's backend.

This notebook focuses exclusively on final model training and deployment, without additional model selection or exploratory analysis.

In [7]:
%pip install shap

  Using cached shap-0.54.1-cp312-abi3-win_amd64.whl.metadata (23 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
Using cached shap-0.54.1-cp312-abi3-win_amd64.whl (751 kB)
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   ---------------------------------------- 3/3 [shap]

Note: you may need to restart the kernel


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

import json
from pathlib import Path

import shap
from sqlalchemy import text

from sklearn.pipeline import Pipeline
from sklearn.compose import (
    ColumnTransformer,
    TransformedTargetRegressor
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from mlflow.models import infer_signature
from mlflow_utils import load_config, test_server_connection
from postgre_tools import Users, create_postgres_engine

# Reproducibility
RANDOM_STATE = 42

# Final feature configuration
FEATURES = [
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "district",
    "amenity_count"
]

NUMERIC_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "amenity_count"
]

CATEGORICAL_FEATURES = [
    "room_type",
    "district"
]

# Previously selected hyperparameters
RF_PARAMS = {
    "n_estimators": 352,
    "max_depth": 10,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "min_samples_split": 7,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}

# MLflow configuration
EXPERIMENT_NAME = "airbnb-price-prediction"

print("=== FINAL MODEL CONFIGURATION ===")
print("Features:", FEATURES)
print("Numerical:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("Target transformation: log1p")
print("Algorithm: RandomForestRegressor")
print("MLflow experiment:", EXPERIMENT_NAME)


=== FINAL MODEL CONFIGURATION ===
Features: ['room_type', 'accommodates', 'bedrooms', 'beds', 'district', 'amenity_count']
Numerical: ['accommodates', 'bedrooms', 'beds', 'amenity_count']
Categorical: ['room_type', 'district']
Target transformation: log1p
Algorithm: RandomForestRegressor
MLflow experiment: airbnb-price-prediction


## 2. Data Retrieval

The Airbnb dataset originates from Inside Airbnb and contains property listings in Stockholm County.

Two PostgreSQL tables are used:

- **`listings`** contains nightly prices, property characteristics, room types, and district information.
- **`listing_amenities`** contains amenities associated with each listing.

The tables are linked using listing identifiers. Amenity count is calculated as the number of distinct amenities associated with each property.

Only listings with valid, positive nightly prices are included in the training dataset.

In [2]:

# Connect to PostgreSQL
engine = create_postgres_engine(Users.AIRBNB.load_user())

# Retrieve listings and amenities
listings_df = pd.read_sql_query(
    text("SELECT * FROM listings"),
    engine
)

listing_amenities_df = pd.read_sql_query(
    text("SELECT listing_id, amenity FROM listing_amenities"),
    engine
)

# Validate required columns
required_columns = {"id", "price", *FEATURES} - {"amenity_count"}

assert required_columns.issubset(listings_df.columns), (
    f"Missing listing columns: {required_columns - set(listings_df.columns)}"
)

assert {"listing_id", "amenity"}.issubset(
    listing_amenities_df.columns
)

assert listings_df["id"].is_unique

# Keep listings with valid prices
priced_df = listings_df.loc[
    listings_df["price"].notna()
    & (listings_df["price"] > 0)
].copy()

# Count distinct amenities per listing
amenity_counts = (
    listing_amenities_df
    .groupby("listing_id")["amenity"]
    .nunique()
    .rename("amenity_count")
)

# Merge datasets
model_df = priced_df.merge(
    amenity_counts,
    left_on="id",
    right_index=True,
    how="left",
    validate="one_to_one"
)

# Listings without amenity records receive a count of zero
model_df["amenity_count"] = (
    model_df["amenity_count"]
    .fillna(0)
    .astype(int)
)

# Prepare final training features and target
X = model_df[FEATURES].copy()
y = model_df["price"].copy()

print("=== TRAINING DATA ===")
print(f"Listings retrieved: {len(listings_df):,}")
print(f"Listings with valid prices: {len(model_df):,}")
print(f"Training observations: {len(X):,}")
print(f"Input features: {X.shape[1]}")

print("\n=== FEATURE TYPES ===")
print(X.dtypes.to_string())

print("\n=== DISTRICT INFORMATION ===")
print(f"Unique districts: {X['district'].nunique()}")
print(f"Missing districts: {X['district'].isna().sum():,}")

print("\nTraining dataset prepared successfully.")


=== TRAINING DATA ===
Listings retrieved: 4,806
Listings with valid prices: 3,264
Training observations: 3,264
Input features: 6

=== FEATURE TYPES ===
room_type            str
accommodates       int64
bedrooms         float64
beds             float64
district             str
amenity_count      int64

=== DISTRICT INFORMATION ===
Unique districts: 11
Missing districts: 0

Training dataset prepared successfully.


## 3. Final Model Training

The final model uses six predefined features and a Random Forest Regressor with hyperparameters established during previous experiments.

Numerical features are processed using median imputation. Categorical features are processed using most-frequent imputation and one-hot encoding.

The target variable is transformed using `log1p` during training and converted back to SEK using `expm1` during prediction.

The complete preprocessing and regression pipeline is trained on all 3,264 listings with valid nightly prices and will be saved to MLflow for deployment.

In [3]:

# === FINAL PREPROCESSING PIPELINE ===

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=10,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES)
    ],
    verbose_feature_names_out=False
)

# === RANDOM FOREST ===

regression_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(**RF_PARAMS))
])

# === LOGARITHMIC TARGET TRANSFORMATION ===

final_model = TransformedTargetRegressor(
    regressor=regression_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# === TRAIN FINAL MODEL ===

final_model.fit(X, y)

print("=== FINAL MODEL TRAINED ===")
print("Algorithm: RandomForestRegressor")
print("Target transformation: log1p")
print(f"Training observations: {len(X):,}")
print(f"Input features: {len(FEATURES)}")
print(f"Unique districts: {X['district'].nunique()}")

# Verify that the model produces valid price predictions
sample_predictions = final_model.predict(X.head(5))

assert np.isfinite(sample_predictions).all()
assert (sample_predictions > 0).all()

print("\n=== PREDICTION VERIFICATION ===")

for i, prediction in enumerate(sample_predictions, start=1):
    print(f"Listing {i}: {prediction:,.2f} SEK/night")

print("\nFinal model training completed successfully.")


=== FINAL MODEL TRAINED ===
Algorithm: RandomForestRegressor
Target transformation: log1p
Training observations: 3,264
Input features: 6
Unique districts: 11

=== PREDICTION VERIFICATION ===
Listing 1: 1,153.50 SEK/night
Listing 2: 1,309.33 SEK/night
Listing 3: 759.74 SEK/night
Listing 4: 857.52 SEK/night
Listing 5: 1,627.87 SEK/night

Final model training completed successfully.


## 4. Model Deployment with MLflow

The trained Random Forest model is stored in MLflow together with its complete preprocessing pipeline and logarithmic target transformation.

The logged model includes an input/output signature, an example prediction request, hyperparameters, feature configuration, and supported categorical values.

MLflow provides a persistent model URI that can be used by the FastAPI backend to load the trained model and generate nightly price predictions in SEK.

The final model is trained on all available listings with valid prices. No further model comparison or hyperparameter optimization is performed.

In [4]:

# === CONNECT TO MLFLOW ===

load_config()
test_server_connection()

mlflow.set_experiment(EXPERIMENT_NAME)

# === PREPARE MODEL METADATA ===

input_example = X.head(5).copy()

signature = infer_signature(
    input_example,
    final_model.predict(input_example)
)

# Valid categorical values for frontend dropdowns
supported_categories = {
    "districts": sorted(
        X["district"].dropna().unique().tolist()
    ),
    "room_types": sorted(
        X["room_type"].dropna().unique().tolist()
    )
}

model_metadata = {
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "target": "price",
    "target_unit": "SEK_per_night",
    "target_transformation": "log1p",
    "training_observations": len(X),
    "supported_categories": supported_categories
}

# === LOG MODEL ===

with mlflow.start_run(
    run_name="airbnb-final-six-feature-model"
) as run:

    mlflow.log_params({
        "algorithm": "RandomForestRegressor",
        "feature_count": len(FEATURES),
        "features": ", ".join(FEATURES),
        "training_observations": len(X),
        "target_transformation": "log1p",
        **RF_PARAMS
    })

    mlflow.log_dict(
        model_metadata,
        "model_metadata.json"
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=final_model,
        name="model",
        signature=signature,
        input_example=input_example,
        serialization_format="skops",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    final_run_id = run.info.run_id
    final_model_uri = model_info.model_uri

# === OUTPUT ===

print("=== MODEL SAVED TO MLFLOW ===")
print("Experiment:", EXPERIMENT_NAME)
print("Run ID:", final_run_id)
print("Model URI:", final_model_uri)

print("\n=== SUPPORTED CATEGORIES ===")
print("Districts:", supported_categories["districts"])
print("Room types:", supported_categories["room_types"])

print("\nModel successfully logged to MLflow.")


Configuration loaded successfully.
MLflow connection successful!
Found 1 experiment(s)


c:\Users\Kim\software engineering\year 5\LP1\4DT911\4dt911-vis-project-research\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run airbnb-final-six-feature-model at: https://cu0089.camp.lnu.se/mlflow/#/experiments/2/runs/0473d5eb85684c94b50f54bf0cba8cc5
🧪 View experiment at: https://cu0089.camp.lnu.se/mlflow/#/experiments/2
=== MODEL SAVED TO MLFLOW ===
Experiment: airbnb-price-prediction
Run ID: 0473d5eb85684c94b50f54bf0cba8cc5
Model URI: models:/m-c32e28368f0342d29635e00a6cab9a61

=== SUPPORTED CATEGORIES ===
Districts: ['Bromma', 'Enskede-Årsta-Vantör', 'Farsta', 'Hägersten-Älvsjö', 'Hässelby-Vällingby', 'Järva', 'Kungsholmen', 'Norra innerstaden', 'Skarpnäck', 'Skärholmen', 'Södermalm']
Room types: ['Entire home/apt', 'Private room', 'Shared room']

Model successfully logged to MLflow.


## 5. Model Verification

The final model is loaded from MLflow and tested using the same input observations as the original trained model.

Predictions are compared to verify that model serialization and retrieval preserve the trained pipeline's behavior.

Successful verification confirms that the saved model is ready for integration with the FastAPI backend.

In [5]:

# === LOAD MODEL FROM MLFLOW ===

loaded_model = mlflow.sklearn.load_model(
    final_model_uri
)

# Use sample observations for verification
verification_input = X.head(10).copy()

# Predict using original and loaded models
original_predictions = final_model.predict(
    verification_input
)

loaded_predictions = loaded_model.predict(
    verification_input
)

# Compare predictions
prediction_difference = np.abs(
    original_predictions - loaded_predictions
)

print("=== MLFLOW MODEL VERIFICATION ===")
print("Model URI:", final_model_uri)
print("Observations tested:", len(verification_input))

print("\n=== PREDICTION COMPARISON ===")

comparison_df = pd.DataFrame({
    "Original": original_predictions,
    "Loaded": loaded_predictions,
    "Difference": prediction_difference
})

print(
    comparison_df
    .round(6)
    .to_string(index=False)
)

# Verify consistency
assert np.allclose(
    original_predictions,
    loaded_predictions,
    rtol=1e-10,
    atol=1e-8
), "Loaded model predictions differ!"

print("\nMaximum prediction difference:",
      prediction_difference.max())

print("\nModel verification successful!")


=== MLFLOW MODEL VERIFICATION ===
Model URI: models:/m-c32e28368f0342d29635e00a6cab9a61
Observations tested: 10

=== PREDICTION COMPARISON ===
   Original      Loaded  Difference
1153.495490 1153.495490         0.0
1309.326489 1309.326489         0.0
 759.736621  759.736621         0.0
 857.521444  857.521444         0.0
1627.871991 1627.871991         0.0
 853.955544  853.955544         0.0
4241.839188 4241.839188         0.0
2104.841437 2104.841437         0.0
 558.795451  558.795451         0.0
 541.888303  541.888303         0.0

Maximum prediction difference: 7.275957614183426e-12

Model verification successful!


## 6. SHAP Explainability

SHAP is used to explain how the six property characteristics contribute to the final model's predicted nightly price.

A model-agnostic permutation explainer is applied to the complete trained prediction pipeline, ensuring that feature contributions are expressed in SEK.

Categorical features (`room_type` and `district`) are encoded using reversible numerical mappings for SHAP calculations.

A reproducible background dataset is sampled from the training observations. The background data and categorical mappings are saved to MLflow so the backend can recreate the explainer for individual predictions.

SHAP explanations are calculated on demand rather than precomputed for every possible property configuration.

In [9]:

# === SHAP CONFIGURATION ===

SHAP_BACKGROUND_SIZE = 30
SHAP_MAX_EVALS = 200

# Create mappings for the two categorical features
category_mappings = {
    feature: {
        category: code
        for code, category in enumerate(
            sorted(X[feature].dropna().unique())
        )
    }
    for feature in CATEGORICAL_FEATURES
}

reverse_category_mappings = {
    feature: {
        code: category
        for category, code in mapping.items()
    }
    for feature, mapping in category_mappings.items()
}


def encode_shap_input(df):
    """Encode categorical inputs for SHAP."""

    encoded = df[FEATURES].copy()

    for feature, mapping in category_mappings.items():
        encoded[feature] = encoded[feature].map(mapping)

    return encoded.astype(float)


def predict_for_shap(values):
    """Convert SHAP inputs into model predictions in SEK."""

    df = pd.DataFrame(
        values,
        columns=FEATURES
    ).copy()

    for feature, mapping in reverse_category_mappings.items():
        df[feature] = [
            mapping.get(int(round(code))) if pd.notna(code)
            else np.nan
            for code in df[feature]
        ]

    return final_model.predict(df[FEATURES])


# Reproducible background from training data
shap_background = encode_shap_input(
    X.sample(
        n=min(SHAP_BACKGROUND_SIZE, len(X)),
        random_state=RANDOM_STATE
    )
)

# Verify that encoding does not change predictions
sample = X.sample(n=5, random_state=123)
encoded_sample = encode_shap_input(sample)

original_predictions = final_model.predict(sample)
wrapped_predictions = predict_for_shap(
    encoded_sample.to_numpy()
)

assert np.allclose(
    original_predictions,
    wrapped_predictions,
    rtol=1e-10,
    atol=1e-8
)

print("=== SHAP CONFIGURATION ===")
print("Features:", FEATURES)
print("Background observations:", len(shap_background))
print("Categorical mappings:")

for feature, mapping in category_mappings.items():
    print(f"  {feature}: {len(mapping)} categories")

print("\nPrediction consistency verified.")


=== SHAP CONFIGURATION ===
Features: ['room_type', 'accommodates', 'bedrooms', 'beds', 'district', 'amenity_count']
Background observations: 30
Categorical mappings:
  room_type: 3 categories
  district: 11 categories

Prediction consistency verified.


In [10]:

# === INITIALIZE SHAP EXPLAINER ===

shap_explainer = shap.Explainer(
    predict_for_shap,
    shap_background,
    algorithm="permutation",
    seed=RANDOM_STATE
)

# Select one listing for verification
shap_test_listing = X.sample(
    n=1,
    random_state=123
)

shap_test_encoded = encode_shap_input(
    shap_test_listing
)

# Calculate SHAP contributions in SEK
shap_test_values = shap_explainer(
    shap_test_encoded,
    max_evals=SHAP_MAX_EVALS
)

# Verify explanation reconstruction
predicted_price = float(
    final_model.predict(shap_test_listing)[0]
)

shap_base_value = float(
    shap_test_values.base_values[0]
)

shap_contributions = shap_test_values.values[0]

reconstructed_price = (
    shap_base_value + shap_contributions.sum()
)

reconstruction_difference = abs(
    predicted_price - reconstructed_price
)

print("=== SHAP VERIFICATION ===")
print(f"Predicted price: {predicted_price:.2f} SEK")
print(f"SHAP baseline: {shap_base_value:.2f} SEK")
print(f"Reconstructed price: {reconstructed_price:.2f} SEK")
print(f"Difference: {reconstruction_difference:.6f} SEK")

print("\n=== FEATURE CONTRIBUTIONS ===")

contributions_df = pd.DataFrame({
    "feature": FEATURES,
    "shap_sek": shap_contributions
})

print(
    contributions_df
    .round(2)
    .to_string(index=False)
)

assert shap_test_values.values.shape == (1, len(FEATURES))
assert reconstruction_difference < 1.0, (
    "SHAP reconstruction differs by at least 1 SEK"
)

print("\nSHAP verification successful!")


=== SHAP VERIFICATION ===
Predicted price: 542.85 SEK
SHAP baseline: 2063.68 SEK
Reconstructed price: 542.85 SEK
Difference: 0.000000 SEK

=== FEATURE CONTRIBUTIONS ===
      feature  shap_sek
    room_type   -364.68
 accommodates   -500.00
     bedrooms   -216.00
         beds    -39.14
     district   -398.85
amenity_count     -2.17

SHAP verification successful!


In [12]:

# === PREPARE SHAP ARTIFACTS ===

ARTIFACT_DIR = Path("models/airbnb_price_prediction")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

background_path = ARTIFACT_DIR / "shap_background.csv"
config_path = ARTIFACT_DIR / "shap_config.json"

# Save the exact background observations used by SHAP
shap_background.to_csv(
    background_path,
    index=False
)

# Preserve everything needed to reconstruct the explainer
shap_config = {
    "model_uri": final_model_uri,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "category_mappings": {
        feature: {
            str(category): int(code)
            for category, code in mapping.items()
        }
        for feature, mapping in category_mappings.items()
    },
    "algorithm": "permutation",
    "max_evals": SHAP_MAX_EVALS,
    "background_samples": len(shap_background),
    "random_state": RANDOM_STATE,
    "output_unit": "SEK",
    "background_file": "shap_background.csv"
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        shap_config,
        f,
        indent=2,
        ensure_ascii=False
    )

# === VALIDATE SAVED ARTIFACTS ===

saved_background = pd.read_csv(background_path)

with open(config_path, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

assert saved_background.shape == shap_background.shape
assert saved_config["features"] == FEATURES
assert saved_config["model_uri"] == final_model_uri

# === LOG TO EXISTING MLFLOW RUN ===

with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact(
        str(background_path),
        artifact_path="shap"
    )

    mlflow.log_artifact(
        str(config_path),
        artifact_path="shap"
    )

print("=== SHAP ARTIFACTS SAVED ===")
print("MLflow run:", final_run_id)
print("Model URI:", final_model_uri)

print("\nUploaded artifacts:")
print("  shap/shap_background.csv")
print("  shap/shap_config.json")

print("\n=== SHAP CONFIGURATION ===")
print("Features:", len(saved_config["features"]))
print("Background observations:", len(saved_background))
print("Algorithm:", saved_config["algorithm"])
print("Output unit:", saved_config["output_unit"])

print("\nSHAP artifacts successfully logged to MLflow.")


🏃 View run airbnb-final-six-feature-model at: https://cu0089.camp.lnu.se/mlflow/#/experiments/2/runs/0473d5eb85684c94b50f54bf0cba8cc5
🧪 View experiment at: https://cu0089.camp.lnu.se/mlflow/#/experiments/2
=== SHAP ARTIFACTS SAVED ===
MLflow run: 0473d5eb85684c94b50f54bf0cba8cc5
Model URI: models:/m-c32e28368f0342d29635e00a6cab9a61

Uploaded artifacts:
  shap/shap_background.csv
  shap/shap_config.json

=== SHAP CONFIGURATION ===
Features: 6
Background observations: 30
Algorithm: permutation
Output unit: SEK

SHAP artifacts successfully logged to MLflow.


## 7. Deployment Verification

The saved model and SHAP artifacts are retrieved from MLflow to verify that the deployment package is self-contained.

The saved categorical mappings and background data are used to reconstruct the SHAP explainer.

A sample property is used to verify that the retrieved model produces a valid nightly price prediction and that its SHAP contributions reconstruct the prediction in SEK.

Successful verification confirms that the FastAPI backend can load the model and generate explainable predictions using the stored MLflow artifacts.

In [13]:

# === DOWNLOAD ARTIFACTS FROM MLFLOW ===

downloaded_config_path = mlflow.artifacts.download_artifacts(
    run_id=final_run_id,
    artifact_path="shap/shap_config.json"
)

downloaded_background_path = mlflow.artifacts.download_artifacts(
    run_id=final_run_id,
    artifact_path="shap/shap_background.csv"
)

with open(downloaded_config_path, "r", encoding="utf-8") as f:
    deployed_config = json.load(f)

deployed_background = pd.read_csv(
    downloaded_background_path
)

# === LOAD SAVED MODEL ===

deployed_model = mlflow.sklearn.load_model(
    deployed_config["model_uri"]
)

deployed_features = deployed_config["features"]

# Restore categorical mappings
deployed_reverse_mappings = {
    feature: {
        int(code): category
        for category, code in mapping.items()
    }
    for feature, mapping
    in deployed_config["category_mappings"].items()
}


# === DEPLOYMENT PREDICTION FUNCTION ===

def deployed_predict_for_shap(values):
    """Predict using only the loaded MLflow model and mappings."""

    df = pd.DataFrame(
        values,
        columns=deployed_features
    ).copy()

    for feature, mapping in deployed_reverse_mappings.items():
        df[feature] = [
            mapping.get(int(round(code)))
            if pd.notna(code) else np.nan
            for code in df[feature]
        ]

    return deployed_model.predict(df[deployed_features])


# === RECONSTRUCT SHAP EXPLAINER ===

deployed_explainer = shap.Explainer(
    deployed_predict_for_shap,
    deployed_background[deployed_features],
    algorithm=deployed_config["algorithm"],
    seed=deployed_config["random_state"]
)

# Use a saved background observation as the test input
test_input = deployed_background[deployed_features].iloc[[0]]

# Predict directly using the deployed wrapper
deployed_prediction = float(
    deployed_predict_for_shap(test_input.to_numpy())[0]
)

# Calculate SHAP contributions
deployed_shap_values = deployed_explainer(
    test_input,
    max_evals=deployed_config["max_evals"]
)

# Reconstruct predicted price
deployed_baseline = float(
    deployed_shap_values.base_values[0]
)

deployed_contributions = deployed_shap_values.values[0]

deployed_reconstruction = (
    deployed_baseline + deployed_contributions.sum()
)

difference = abs(
    deployed_prediction - deployed_reconstruction
)

# === VERIFY ===

assert deployed_shap_values.values.shape == (
    1,
    len(deployed_features)
)

assert difference < 1.0, (
    "SHAP reconstruction differs by at least 1 SEK"
)

print("=== MLFLOW DEPLOYMENT VERIFICATION ===")
print("Model URI:", deployed_config["model_uri"])
print("Features:", deployed_features)
print("Background observations:", len(deployed_background))

print("\n=== PREDICTION AND SHAP ===")
print(f"Predicted price: {deployed_prediction:.2f} SEK")
print(f"SHAP baseline: {deployed_baseline:.2f} SEK")
print(f"Reconstructed price: {deployed_reconstruction:.2f} SEK")
print(f"Difference: {difference:.6f} SEK")

print("\n=== SHAP CONTRIBUTIONS ===")

print(
    pd.DataFrame({
        "feature": deployed_features,
        "shap_sek": deployed_contributions
    }).round(2).to_string(index=False)
)

print("\nDeployment verification successful!")


=== MLFLOW DEPLOYMENT VERIFICATION ===
Model URI: models:/m-c32e28368f0342d29635e00a6cab9a61
Features: ['room_type', 'accommodates', 'bedrooms', 'beds', 'district', 'amenity_count']
Background observations: 30

=== PREDICTION AND SHAP ===
Predicted price: 3135.58 SEK
SHAP baseline: 2063.68 SEK
Reconstructed price: 3135.58 SEK
Difference: 0.000000 SEK

=== SHAP CONTRIBUTIONS ===
      feature  shap_sek
    room_type    140.00
 accommodates    705.10
     bedrooms    492.43
         beds     87.90
     district   -325.78
amenity_count    -27.74

Deployment verification successful!


## 8. Example: Model Inference from MLflow

This section demonstrates how an external application can retrieve the trained Airbnb price prediction model and SHAP artifacts directly from MLflow.

A sample Airbnb property is provided using the six required input features. The saved model generates a nightly price prediction in SEK, while the reconstructed SHAP explainer calculates the contribution of each feature.

This example follows the same workflow intended for integration with the FastAPI backend and does not require access to the training dataset.

In [14]:

# === 1. CONNECT TO MLFLOW ===

load_config()

MODEL_URI = "models:/m-c32e28368f0342d29635e00a6cab9a61"
RUN_ID = "0473d5eb85684c94b50f54bf0cba8cc5"

# Load trained model
model = mlflow.sklearn.load_model(MODEL_URI)

# Download SHAP artifacts
config_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path="shap/shap_config.json"
)

background_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path="shap/shap_background.csv"
)

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

background = pd.read_csv(background_path)

assert config["model_uri"] == MODEL_URI

# === 2. DEFINE EXAMPLE AIRBNB PROPERTY ===

property_data = pd.DataFrame([{
    "room_type": "Entire home/apt",
    "accommodates": 4,
    "bedrooms": 2.0,
    "beds": 3.0,
    "district": "Södermalm",
    "amenity_count": 20
}])

features = config["features"]
property_data = property_data[features]

# === 3. PREDICT NIGHTLY PRICE ===

predicted_price = float(model.predict(property_data)[0])

# === 4. RECREATE SHAP PREDICTION WRAPPER ===

category_mappings = config["category_mappings"]

reverse_mappings = {
    feature: {
        int(code): category
        for category, code in mapping.items()
    }
    for feature, mapping in category_mappings.items()
}


def predict_with_shap(values):
    df = pd.DataFrame(values, columns=features).copy()

    for feature, mapping in reverse_mappings.items():
        df[feature] = [
            mapping.get(int(round(code)))
            if pd.notna(code) else np.nan
            for code in df[feature]
        ]

    return model.predict(df[features])


# Encode the new property for SHAP
encoded_property = property_data.copy()

for feature, mapping in category_mappings.items():
    encoded_property[feature] = encoded_property[feature].map(
        mapping
    )

encoded_property = encoded_property.astype(float)

# === 5. GENERATE SHAP EXPLANATION ===

explainer = shap.Explainer(
    predict_with_shap,
    background[features],
    algorithm=config["algorithm"],
    seed=config["random_state"]
)

explanation = explainer(
    encoded_property,
    max_evals=config["max_evals"]
)

base_value = float(explanation.base_values[0])
contributions = explanation.values[0]

# Verify prediction reconstruction
reconstructed_price = base_value + contributions.sum()

assert np.isclose(
    predicted_price,
    reconstructed_price,
    atol=1.0
)

# === 6. DISPLAY RESULTS ===

print("=== EXAMPLE AIRBNB PROPERTY ===")
print(property_data.to_string(index=False))

print("\n=== PRICE PREDICTION ===")
print(f"Predicted nightly price: {predicted_price:,.2f} SEK")

print("\n=== SHAP EXPLANATION ===")
print(f"Baseline: {base_value:,.2f} SEK")

shap_results = pd.DataFrame({
    "feature": features,
    "value": property_data.iloc[0].values,
    "contribution_sek": contributions
})

print(
    shap_results
    .round(2)
    .to_string(index=False)
)

print("\n=== RECONSTRUCTION ===")
print(f"Reconstructed price: {reconstructed_price:,.2f} SEK")
print(f"Difference: {abs(predicted_price - reconstructed_price):.6f} SEK")


Configuration loaded successfully.


=== EXAMPLE AIRBNB PROPERTY ===
      room_type  accommodates  bedrooms  beds  district  amenity_count
Entire home/apt             4       2.0   3.0 Södermalm             20

=== PRICE PREDICTION ===
Predicted nightly price: 3,047.36 SEK

=== SHAP EXPLANATION ===
Baseline: 2,063.68 SEK
      feature           value  contribution_sek
    room_type Entire home/apt            211.21
 accommodates               4            162.56
     bedrooms             2.0             63.93
         beds             3.0             28.49
     district       Södermalm            601.09
amenity_count              20            -83.59

=== RECONSTRUCTION ===
Reconstructed price: 3,047.36 SEK
Difference: 0.000000 SEK
